# IndexTTS-2.5 · Google Colab Pro L4

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zencolab/colab/blob/main/IndexTTS2_5_L4_Colab.ipynb)

模型：**IndexTeam/IndexTTS-2.5**（目前官方最新模型）。代码默认固定到官方 `main` 最新提交 `d9e41aa`（包含 v2.5.0 之后的修复：结尾咔哒声淡出、`do_sample` 透传、推理提速、WebUI 预设/`--share` 等），也可改回 `v2.5.0`。

**本版修复的 Colab 报错**：Colab 预设了 `UV_PRERELEASE=`（空值）、`UV_CONSTRAINT`、`UV_SYSTEM_PYTHON`、`MPLBACKEND=module://matplotlib_inline...`、`PYTHONPATH` 等环境变量，会导致 `uv sync` 直接报 `a value is required for '--prerelease'`，或在 uv 虚拟环境里导入 matplotlib 失败。现在所有 `uv` 命令都在清理后的环境中运行。

> 先在「运行时 → 更改运行时类型」选择 GPU/L4。只使用本人或已获明确授权的声音，并阅读上游许可与免责声明。


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

#@markdown ### 基本设置
CODE_REF = "d9e41aac89fd00b3d71497fddb287b7f24613712" #@param {type:"string"}
#@markdown ↑ 官方仓库的提交 SHA 或标签（如 `v2.5.0`）
USE_DRIVE = False #@param {type:"boolean"}
HF_ENDPOINT = "" #@param {type:"string"}
REPO = Path("/content/index-tts")

# Colab 注入的这些变量会破坏 uv / 子进程虚拟环境，统一清理
_BAD_PREFIX = ("UV_",)
_BAD_KEYS = {"MPLBACKEND", "PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "CONDA_PREFIX", "PIP_CONSTRAINT"}
def clean_env():
    env = {k: v for k, v in os.environ.items() if k not in _BAD_KEYS and not k.startswith(_BAD_PREFIX)}
    env.update(UV_LINK_MODE="copy", MPLBACKEND="Agg", PYTHONUNBUFFERED="1",
               HF_HUB_DISABLE_PROGRESS_BARS="0", TOKENIZERS_PARALLELISM="false")
    if HF_ENDPOINT.strip():
        env["HF_ENDPOINT"] = HF_ENDPOINT.strip().rstrip("/")
    return env

def run(cmd, cwd=None):
    cmd = list(map(str, cmd))
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=clean_env(), check=True)

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     text=True, capture_output=True)
if smi.returncode:
    raise RuntimeError("未检测到 NVIDIA GPU，请切换到 Colab GPU 运行时。")
print(smi.stdout.strip())
if "L4" not in smi.stdout:
    print("⚠️ 当前不是 L4，仍可尝试运行（建议显存 ≥ 16GB）。")


## 1. 安装

首次安装约 3–6 分钟。uv 会自动下载项目要求的 Python 3.11 并按官方 `uv.lock` 安装（CUDA 12.8 版 PyTorch 2.8）。不安装易在 Colab 编译失败的 DeepSpeed/FlashAttention。


In [ ]:
run(["apt-get", "-qq", "update"])
run(["apt-get", "-qq", "install", "-y", "git", "ffmpeg"])

if not (REPO / ".git").exists():
    if REPO.exists(): shutil.rmtree(REPO)
    run(["git", "init", "-q", REPO])
    run(["git", "remote", "add", "origin", "https://github.com/index-tts/index-tts.git"], REPO)
run(["git", "fetch", "-q", "--depth", "1", "origin", CODE_REF], REPO)
run(["git", "checkout", "-q", "--force", "FETCH_HEAD"], REPO)
run(["git", "log", "-1", "--format=%h %cs %s"], REPO)

run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"])
UV = shutil.which("uv") or "/usr/local/bin/uv"
run([UV, "--version"])
run([UV, "sync", "--extra", "webui", "--frozen"], REPO)
run([UV, "run", "--no-sync", "python", "-c",
     "import torch;print('torch',torch.__version__,'cuda',torch.cuda.is_available())"], REPO)


## 2. 下载 IndexTTS-2.5 模型

开启 `USE_DRIVE` 后模型保存到 `MyDrive/AI-Models/IndexTTS-2.5`，下次会话可直接复用。


In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    MODEL_DIR = Path("/content/drive/MyDrive/AI-Models/IndexTTS-2.5")
else:
    MODEL_DIR = REPO / "checkpoints"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

program = f"""
from indextts.utils.model_download import snapshot_download, ensure_config_available, ensure_models_available
m = {str(MODEL_DIR)!r}
snapshot_download("IndexTeam/IndexTTS-2.5", local_dir=m)
ensure_config_available(m, version="2.5")
ensure_models_available(m)
"""
run([UV, "run", "--no-sync", "python", "-c", program], REPO)

required = ["config.yaml", "gpt.pth", "s2mel.pth", "codec.pth",
            "multilingual_zh_ja_yue_char_del.tiktoken", "wav2vec2bert_stats.pt"]
missing = [x for x in required if not (MODEL_DIR / x).is_file()]
if missing:
    raise FileNotFoundError("模型不完整，请重跑本单元格：" + ", ".join(missing))
run([UV, "run", "--no-sync", "python", "tools/gpu_check.py"], REPO)
print("✅ 模型就绪：", MODEL_DIR)


## 3. 启动官方 WebUI（推荐）

运行完第 1、2 节后直接运行本单元格即可，在网页里上传参考音频并生成，无需运行第 4 节。等待输出 `https://xxxx.gradio.live` 链接。单元格会持续运行，用完点击停止。L4（24GB）会额外加载 QwenEmotion 文本情感模型，支持“用文本描述情感”。


In [ ]:
LAUNCH_WEBUI = True #@param {type:"boolean"}
if LAUNCH_WEBUI:
    run([UV, "run", "--no-sync", "python", "webui.py", "--version", "2.5", "--model_dir", MODEL_DIR,
         "--fp16", "--share", "--host", "0.0.0.0", "--port", 7860], REPO)
else:
    print("已跳过 WebUI")


## 4. 可选：命令行推理（不用 WebUI 时才运行）

使用上面的 WebUI 时可**跳过本节全部单元格**。建议 5–15 秒、单人、无音乐/混响的清晰音频（wav/mp3/m4a/flac 均可，会自动转成 WAV）。`LANG` 支持 `ZH/EN/JA/ES/AR`；`DURATION_FACTOR<1` 语速更快，`>1` 更慢。情感向量顺序：`[高兴,愤怒,悲伤,害怕,厌恶,忧郁,惊讶,平静]`，例如 `0.6,0,0,0,0,0,0.2,0.2`。


In [ ]:
helper = REPO / "colab_infer_v25.py"
helper.write_text('#!/usr/bin/env python3\n"""Memory-conscious IndexTTS-2.5 inference for Google Colab (L4)."""\nimport argparse, json, random\nfrom pathlib import Path\nimport numpy as np, torch\nfrom indextts.infer_v2_5 import IndexTTS2\n\ndef parse_vector(raw):\n    raw = (raw or "").strip()\n    if not raw:\n        return None\n    values = json.loads(raw) if raw.startswith("[") else [x for x in raw.replace("，", ",").split(",")]\n    values = [float(x) for x in values]\n    if len(values) != 8 or any(not 0 <= x <= 1 for x in values):\n        raise ValueError("emo_vector 需要 8 个 0~1 之间的数字")\n    return values\n\np = argparse.ArgumentParser()\np.add_argument("--model_dir", required=True)\np.add_argument("--prompt_wav", required=True)\np.add_argument("--text", required=True)\np.add_argument("--lang", choices=["ZH", "EN", "JA", "ES", "AR"], default="ZH")\np.add_argument("--output", required=True)\np.add_argument("--duration_factor", type=float, default=1.0)\np.add_argument("--emo_vector", default="")\np.add_argument("--emo_audio", default="")\np.add_argument("--emo_alpha", type=float, default=0.8)\np.add_argument("--seed", type=int, default=1234)\na = p.parse_args()\nif not 0.5 <= a.duration_factor <= 2.0: raise ValueError("duration_factor 需在 0.5~2.0")\nif not 0.0 <= a.emo_alpha <= 1.0: raise ValueError("emo_alpha 需在 0~1")\n\nrandom.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)\nif torch.cuda.is_available(): torch.cuda.manual_seed_all(a.seed)\nuse_bf16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())\nprint("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", "| BF16:", use_bf16)\n\nm = Path(a.model_dir)\ntts = IndexTTS2(cfg_path=str(m / "config.yaml"), model_dir=str(m), use_bf16=use_bf16,\n                use_cuda_kernel=False, use_deepspeed=False, use_accel=False,\n                use_torch_compile=False, use_qwen_emo=False)\nvec = parse_vector(a.emo_vector)\nif vec is not None:\n    vec = tts.normalize_emo_vec(vec, apply_bias=True)\nout = Path(a.output); out.parent.mkdir(parents=True, exist_ok=True)\ntts.infer(spk_audio_prompt=a.prompt_wav, text=a.text, lang=a.lang, output_path=str(out),\n          emo_audio_prompt=a.emo_audio or None, emo_vector=vec, emo_alpha=a.emo_alpha,\n          use_random=False, duration_factor=a.duration_factor, verbose=True)\nprint("Saved:", out.resolve())\n', encoding="utf-8")

from google.colab import files
print("上传已获授权的参考音频：")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("未上传音频")
name, data = next(iter(uploaded.items()))
inputs = REPO / "inputs"; inputs.mkdir(exist_ok=True)
raw = inputs / Path(name).name; raw.write_bytes(data)
REF = inputs / "reference.wav"
run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw, "-ac", "1", "-ar", "24000", REF])
print("参考音频：", REF)


In [ ]:
TEXT = "大家好，这是在 Google Colab L4 GPU 上运行的 IndexTTS 二点五测试。" #@param {type:"string"}
LANG = "ZH" #@param ["ZH","EN","JA","ES","AR"]
DURATION_FACTOR = 1.0 #@param {type:"slider", min:0.5, max:2.0, step:0.05}
EMO_VECTOR = "" #@param {type:"string"}
EMO_ALPHA = 0.8 #@param {type:"slider", min:0, max:1, step:0.05}
SEED = 1234 #@param {type:"integer"}

import time
OUTPUT = REPO / "outputs" / f"indextts2_5_{time.strftime('%Y%m%d_%H%M%S')}.wav"
cmd = [UV, "run", "--no-sync", "python", helper, "--model_dir", MODEL_DIR, "--prompt_wav", REF,
       "--text", TEXT, "--lang", LANG, "--output", OUTPUT,
       "--duration_factor", DURATION_FACTOR, "--emo_alpha", EMO_ALPHA, "--seed", SEED]
if EMO_VECTOR.strip():
    cmd += ["--emo_vector", EMO_VECTOR]
run(cmd, REPO)

from IPython.display import Audio, display
display(Audio(filename=str(OUTPUT)))


In [ ]:
from google.colab import files
files.download(str(OUTPUT))


## 发音与故障排查

- 中文多音字：`他在银<行|XING2>里<行|HANG2>走。`
- 英文：`a <minute|M IH1 . N AH0 T>`；日语：`<上手|じょうず>`
- `a value is required for '--prerelease'` / `requirements.constraints` 找不到：说明运行了旧版 Notebook，请使用本版（已清理 Colab 的 `UV_*` 变量）。
- 下载中断：重跑模型单元格；网络受限可设置 `HF_ENDPOINT=https://hf-mirror.com`。
- OOM：停止 WebUI、重启运行时、缩短文本。
- 想回到正式发布版：把 `CODE_REF` 改成 `v2.5.0`。

来源：[官方仓库](https://github.com/index-tts/index-tts) · [中文文档](https://github.com/index-tts/index-tts/blob/main/docs/README_zh.md) · [模型](https://huggingface.co/IndexTeam/IndexTTS-2.5) · [当前固定提交](https://github.com/index-tts/index-tts/commit/d9e41aac89fd00b3d71497fddb287b7f24613712)。本 Notebook 不分发模型权重。
